# Chocolates e Páscoa — análise exploratória para gerar insights

## Como acompanhar esta aula

Este notebook é um roteiro guiado para iniciantes. Leia cada explicação, tente prever o resultado, execute a célula e diga o que o resultado significa. Execute de cima para baixo, usando um kernel Python com Pandas instalado.

**Preparação:** baixe também `pascoa_vendas.csv` e coloque-o na mesma pasta deste notebook. Abra essa pasta no VS Code. O arquivo já vem pronto: não é necessário programar sua geração. Todos os registros são fictícios e não contêm clientes reais.

**Objetivo:** entender uma loja, investigar a qualidade dos dados e produzir conclusões com evidência. Vamos explorar produto, canal, data, quantidade, receita e margem bruta simulada.

**Tempo sugerido:** 2 horas para leitura, qualidade, timestamps, receita e uma comparação; reserve mais 1 hora, ou uma aula seguinte, para margem, comparações por canal e exercícios. O roteiro completo não precisa ser apresentado de uma só vez.

## 1. Começar pelo negócio, antes do código

A **Doce Estação** é uma rede fictícia que vende chocolates por três canais:

- **Mercado:** vendas registradas nos caixas de supermercados parceiros, para consumidores finais; não são pedidos de atacado.
- **Loja física:** vendas nas lojas próprias.
- **Web:** pedidos feitos pelo site.

A gestão precisa decidir quais produtos priorizar, quando reforçar o estoque e como distribuir a oferta entre os canais. Perguntas: vender mais unidades também significa gerar mais receita? Qual canal concentra ovos especiais? O que continua vendendo depois da Páscoa?

### Uma publicação real para abrir a conversa

Na publicação [Páscoa 2025: Alta de preço do chocolate não deve causar retração no consumo, mantendo estabilidade em relação a 2024](https://www.kantar.com/brazil/inspiration/consumo/2025/consumo-de-chocolates-no-brasil), a Kantar discute formatos de chocolate, escolhas do consumidor e canais de compra no contexto da data. Use essa leitura para motivar perguntas sobre o negócio.

**Perguntas de abertura:** se o consumidor pode escolher uma barra, uma caixa ou um ovo, devemos oferecer a mesma combinação em todos os canais? Como investigar isso com dados?

A publicação trata do mercado real; os CSVs desta atividade são simulações independentes. Seus números não foram usados para reproduzir a pesquisa. O dataset não confirma nem refuta a publicação.

Neste exercício, a Páscoa ocorre em 31/03/2024 e 20/04/2025. Comparar março de um ano com março do outro pode confundir mudança de calendário com mudança de demanda. Vamos retornar a esse cuidado na análise.

## 2. O que é análise exploratória?

É investigar um conjunto de dados para entender sua estrutura, qualidade e padrões antes de concluir ou construir um modelo. Aqui, a sequência será: pergunta de negócio → leitura → inspeção → diagnóstico → organização → cálculos → comparação → insight.

Um **dataset** é o conjunto de dados; um **registro** é uma linha; uma **coluna** descreve uma característica. Um **DataFrame** é uma tabela do Pandas; uma **Series** pode representar uma coluna.

**Neste dataset, uma linha representa um pedido de um único produto.** Cada pedido pode conter várias unidades desse produto. Não representa uma pessoa nem uma unidade individual. Portanto, número de linhas, número de pedidos e número de unidades são medidas diferentes.

Há 1.500 pedidos distintos, distribuídos em dois anos: janeiro a junho de 2024 e janeiro a junho de 2025. O CSV tem 1.515 linhas, incluindo repetições intencionais. Não é uma amostra real do mercado brasileiro.

## 3. Dicionário de dados

| Coluna | Significado | Unidade ou exemplo |
|---|---|---|
| pedido_id | Identificador do pedido | 123; não é uma medida para somar |
| data_venda | Momento em que o pedido foi registrado | `2025-04-12 14:35:00` |
| produto | Nome do item | Ovo premium |
| categoria | Formato do produto | Barra, Caixa ou Ovo |
| canal_venda | Canal em que foi registrado | Mercado, Loja física ou Web |
| cidade | Cidade do pedido | Curitiba |
| quantidade | Unidades do produto no pedido | Inteiro maior que zero |
| preco_unitario | Preço de uma unidade antes do desconto | Reais |
| custo_unitario | Custo de aquisição de uma unidade | Reais |
| desconto_pct | Percentual aplicado ao pedido | 10 significa 10%, não 0,10 |
| status_pedido | Situação registrada | Concluído, Cancelado ou Devolvido |
| tipo_chocolate | Tipo do chocolate | Ao leite, Amargo ou Misto |
| peso_g | Peso de uma unidade | Gramas |

Datas e horas representam horário local de São Paulo, sem indicação de fuso no arquivo. Não converteremos fusos nesta aula. Não há atualização histórica de status: usamos a situação registrada no CSV.

## 4. Ler o arquivo e entender a sintaxe

`import pandas as pd` carrega a biblioteca com o apelido `pd`. O ponto em `pd.read_csv` acessa uma função da biblioteca. Os parênteses chamam a função; o caminho entre aspas é um argumento. O sinal `=` guarda o resultado em uma variável.

`df` é um nome escolhido, não uma palavra obrigatória. `head()` é um método: uma operação acessada pela tabela. `shape` é um atributo: uma informação consultada sem parênteses. Os colchetes de `df["produto"]` selecionam uma coluna.

O trecho de conferência abaixo mostra a pasta de execução. Se o arquivo não existir ali, confirme o local onde foi salvo, o nome e a extensão. Não altere o caminho ao acaso.

In [ ]:
from pathlib import Path
import pandas as pd

arquivo = Path("pascoa_vendas.csv")
print("Pasta de execução:", Path.cwd())
print("Arquivo encontrado:", arquivo.exists())

df = pd.read_csv(arquivo)
df.head()

,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,tipo_chocolate,peso_g
0,1359,2025-05-12 13:22:50,Barra amarga,Barra,Loja física,Maringá,2,11.66,8.16,15,Concluído,Amargo,90
1,550,2024-02-29 18:21:51,Barra ao leite,Barra,Web,Curitiba,3,8.00,5.80,0,Concluído,Ao leite,90
2,654,2024-03-05 21:56:23,Caixa de bombons,Caixa,Web,Curitiba,1,24.00,15.50,0,Concluído,Misto,250
3,1123,2025-04-29 16:01:03,Barra amarga,Barra,Mercado,Curitiba,4,11.66,8.16,10,Concluído,Amargo,90
4,674,2024-04-07 16:58:53,Caixa de bombons,Caixa,Mercado,Cascavel,6,24.00,15.50,25,Concluído,Misto,250


**Leia o resultado:** quais colunas aparecem? Uma linha mostra unidades ou pedidos? Você consegue identificar um preço, um custo e uma data com hora?

`head()` mostra cinco linhas por padrão. É uma inspeção inicial, não uma prova de qualidade da tabela inteira. No notebook, a última expressão da célula costuma ser exibida como tabela. `print()` também pode apresentar os resultados.

In [2]:
print("Linhas e colunas:", df.shape)
print("Nomes das colunas:", list(df.columns))
print(df.dtypes)
df.info()

Linhas e colunas: (1515, 13)
Nomes das colunas: ['pedido_id', 'data_venda', 'produto', 'categoria', 'canal_venda', 'cidade', 'quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'status_pedido', 'tipo_chocolate', 'peso_g']
pedido_id           int64
data_venda         object
produto            object
categoria          object
canal_venda        object
cidade             object
quantidade          int64
preco_unitario    float64
custo_unitario    float64
desconto_pct        int64
status_pedido      object
tipo_chocolate     object
peso_g              int64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1515 entries, 0 to 1514
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       1515 non-null   int64  
 1   data_venda      1515 non-null   object 
 2   produto         1515 non-null   object 
 3   categoria       1515 non-null   object 
 4   canal_venda     1515 non-null 

Esperado: 1.515 linhas e 13 colunas. `shape[0]` consulta a quantidade de linhas; `shape[1]`, de colunas. Os índices começam em zero.

`dtypes` mostra os tipos reconhecidos. Uma coluna que parece data pode ter sido lida como texto. Ter caracteres de ano e mês não a torna automaticamente um datetime. `info()` apresenta tipos e contagens não ausentes; não precisa estar dentro de `print()`.

**Pergunta:** podemos dizer que existem 1.515 pedidos distintos? Ainda não: precisamos verificar as repetições.

## 5. Diagnosticar a qualidade antes de alterar

`isna()` marca ausências com `True`. `sum()` conta esses valores verdadeiros por coluna. Não estamos somando preços. `duplicated()` compara linhas completas e marca repetições posteriores à primeira. Com `keep=False`, mostra todas as ocorrências dos grupos repetidos.

Uma compra do mesmo produto não é necessariamente duplicata. Neste exercício, os registros repetidos têm identificador e campos iguais.

In [3]:
print("Ausências por coluna:")
print(df.isna().sum())
print("Repetições exatas:", df.duplicated().sum())
print(df.loc[df.duplicated(keep=False)].head())
print("Canais como vieram no arquivo:")
print(df["canal_venda"].value_counts())

Ausências por coluna:
pedido_id          0
data_venda         0
produto            0
categoria          0
canal_venda        0
cidade             0
quantidade         0
preco_unitario    12
custo_unitario     7
desconto_pct       0
status_pedido      0
tipo_chocolate     0
peso_g             0
dtype: int64
Repetições exatas: 15
     pedido_id           data_venda  ... tipo_chocolate peso_g
34          11  2024-03-24 21:08:33  ...         Amargo    350
186         15  2024-04-10 16:16:45  ...         Amargo    350
262          4  2024-03-08 09:51:01  ...         Amargo    350
267          8  2024-06-12 21:36:40  ...       Ao leite     90
289          3  2024-03-10 20:49:57  ...       Ao leite    150

[5 rows x 13 columns]
Canais como vieram no arquivo:
canal_venda
Mercado          630
Web              442
Loja física      398
 mercado          17
 web              16
 loja física      12
Name: count, dtype: int64


Um preço ausente não é preço zero. Zero é um valor informado; ausência significa informação desconhecida. Espaços e letras minúsculas também podem separar valores equivalentes de canal.

Vamos trabalhar em uma cópia, retirar somente as repetições exatas confirmadas e padronizar os canais. `str.strip()` retira espaços das bordas; `str.lower()` converte o texto para minúsculas; `map()` aplica um dicionário de correspondências. Um valor desconhecido vira ausente e deve ser investigado, não escondido.

In [4]:
df_org = df.drop_duplicates().copy()

mapa_canais = {
    "mercado": "Mercado",
    "loja física": "Loja física",
    "web": "Web",
}
df_org["canal_venda"] = (
    df_org["canal_venda"].str.strip().str.lower().map(mapa_canais)
)

print("Original:", df.shape)
print("Sem repetições:", df_org.shape)
print(df_org["canal_venda"].value_counts(dropna=False))
print("IDs ainda repetidos:", df_org["pedido_id"].duplicated().sum())

Original: (1515, 13)
Sem repetições: (1500, 13)
canal_venda
Mercado        641
Web            454
Loja física    405
Name: count, dtype: int64
IDs ainda repetidos: 0


Depois de retirar as repetições, temos 1.500 pedidos e nenhum ID repetido. Se um mesmo ID tivesse dados diferentes, precisaríamos investigar qual registro usar; retirar linhas inteiras repetidas não resolveria esse problema.

O arquivo original continua preservado. A variável `df_org` será nossa tabela organizada.

## 6. Timestamp: data e hora são mais que um texto

Um timestamp representa um momento no tempo. Exemplo: `2025-04-12 14:35:00` contém ano, mês, dia, hora, minuto e segundo. No CSV, ele chega como texto; no Pandas, vamos convertê-lo para datetime.

`pd.to_datetime()` faz essa conversão. O formato `%Y-%m-%d %H:%M:%S` descreve ano com quatro dígitos, mês, dia, hora no formato de 24 horas, minuto e segundo. `errors="coerce"` transforma valores que não seguem o formato em `NaT`, a ausência de data/hora. Isso facilita a inspeção, mas exige contar e mostrar as falhas.

Guardaremos o texto original numa coluna adicional para saber quais valores não foram convertidos. Não substituiremos datas inválidas pela data de hoje.

In [5]:
df_org["data_venda_original"] = df_org["data_venda"]
df_org["data_venda"] = pd.to_datetime(
    df_org["data_venda"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

print("Datas inválidas ou ausentes:", df_org["data_venda"].isna().sum())
print(df_org.loc[
    df_org["data_venda"].isna(),
    ["pedido_id", "data_venda_original", "data_venda"],
])

Datas inválidas ou ausentes: 4
      pedido_id data_venda_original data_venda
353         370       data_invalida        NaT
862        1480       data_invalida        NaT
1255       1110       data_invalida        NaT
1306        740       data_invalida        NaT


Há quatro datas inválidas nesta simulação. O comando as marca para investigação. Esses registros não serão usados nas análises por tempo.

### Extrair partes da data com `.dt`

`.dt` dá acesso a operações de data e hora em uma Series de datetime. `.dt.year` extrai ano; `.dt.month`, mês; `.dt.day`, dia do mês; `.dt.hour`, hora. `astype("Int64")` permite inteiros com ausências, representadas por `<NA>`.

Não usaremos `split()` nem recortes de texto para isso. Converter corretamente permite ordenar e calcular diferenças entre datas.

In [6]:
df_org["ano"] = df_org["data_venda"].dt.year.astype("Int64")
df_org["mes"] = df_org["data_venda"].dt.month.astype("Int64")
df_org["dia"] = df_org["data_venda"].dt.day.astype("Int64")
df_org["hora"] = df_org["data_venda"].dt.hour.astype("Int64")
df_org["ano_mes"] = df_org["data_venda"].dt.to_period("M")

print(df_org[["data_venda", "ano", "mes", "dia", "hora", "ano_mes"]].head(8))

           data_venda   ano  mes  dia  hora  ano_mes
0 2025-05-12 13:22:50  2025    5   12    13  2025-05
1 2024-02-29 18:21:51  2024    2   29    18  2024-02
2 2024-03-05 21:56:23  2024    3    5    21  2024-03
3 2025-04-29 16:01:03  2025    4   29    16  2025-04
4 2024-04-07 16:58:53  2024    4    7    16  2024-04
5 2024-02-29 19:19:50  2024    2   29    19  2024-02
6 2025-03-17 18:21:26  2025    3   17    18  2025-03
7 2024-02-27 20:31:44  2024    2   27    20  2024-02


`to_period("M")` representa o mês inteiro, como `2025-04`. Uma hora específica vira uma categoria temporal mensal.

**Cuidado:** agrupar apenas por `mes` reúne abril de 2024 e abril de 2025. Para uma sequência cronológica, use `ano_mes` ou as colunas `ano` e `mes` juntas. Como cada ano tem só janeiro a junho, não chamaremos esses dados de vendas anuais completas.

In [7]:
pedidos_por_periodo = (
    df_org.dropna(subset=["data_venda"])
    .groupby("ano_mes").size()
)
print(pedidos_por_periodo)

print("Pedidos registrados em abril, separados por ano:")
print(df_org.loc[df_org["mes"] == 4].groupby("ano").size())

ano_mes
2024-01     45
2024-02     57
2024-03    371
2024-04    170
2024-05     50
2024-06     55
2025-01     42
2025-02     43
2025-03    157
2025-04    402
2025-05     70
2025-06     34
Freq: M, dtype: int64
Pedidos registrados em abril, separados por ano:
ano
2024    170
2025    402
dtype: int64


A contagem acima inclui todos os status, portanto significa **pedidos registrados**, não vendas concluídas. `size()` conta linhas de cada grupo. Não exige que uma coluna de preço esteja preenchida.

O resultado pode sugerir um período de maior atividade, mas o mês da Páscoa muda de um ano para outro. Precisamos de contexto para interpretar o calendário.

## 7. Conferir medidas e regras do negócio

Nossas regras didáticas: quantidade inteira maior que zero; preço e custo preenchidos e não negativos; desconto preenchido entre 0 e 50%; canal conhecido; status conhecido. O intervalo do desconto é uma regra desta loja fictícia, não de todo comércio.

`pd.to_numeric(..., errors="coerce")` converte para número e transforma texto inválido em ausência. Compare as ausências antes e depois. `notna()` testa preenchimento; `between()` verifica intervalo; `% 1 == 0` verifica se a quantidade é inteira. Os parênteses organizam as condições. `&` combina condições por linha; não use `and` entre Series.

In [8]:
colunas_numericas = ["quantidade", "preco_unitario", "custo_unitario", "desconto_pct"]
antes = df_org[colunas_numericas].isna().sum()
for coluna in colunas_numericas:
    df_org[coluna] = pd.to_numeric(df_org[coluna], errors="coerce")
print("Novas ausências após converter números:")
print(df_org[colunas_numericas].isna().sum() - antes)

status_validos = ["Concluído", "Cancelado", "Devolvido"]
df_org["quantidade_valida"] = (
    df_org["quantidade"].notna()
    & (df_org["quantidade"] > 0)
    & (df_org["quantidade"] % 1 == 0)
)
df_org["valores_validos"] = (
    df_org["preco_unitario"].notna()
    & (df_org["preco_unitario"] >= 0)
    & df_org["custo_unitario"].notna()
    & (df_org["custo_unitario"] >= 0)
    & df_org["desconto_pct"].between(0, 50)
)
df_org["registro_valido"] = (
    df_org["quantidade_valida"]
    & df_org["valores_validos"]
    & df_org["data_venda"].notna()
    & df_org["canal_venda"].notna()
    & df_org["status_pedido"].isin(status_validos)
)
print("Quantidade inválida:", (~df_org["quantidade_valida"]).sum())
print("Registros válidos:", df_org["registro_valido"].value_counts())

Novas ausências após converter números:
quantidade        0
preco_unitario    0
custo_unitario    0
desconto_pct      0
dtype: int64
Quantidade inválida: 7
Registros válidos: registro_valido
True     1470
False      30
Name: count, dtype: int64


O dataset contém 12 preços ausentes, 7 custos ausentes, 7 quantidades inválidas e 4 datas inválidas após remover duplicatas. Os números se sobrepõem em alguns pedidos: não some problemas para presumir quantos registros foram excluídos.

Mantemos os registros problemáticos na base organizada, com flags de validação. Assim, podemos mostrar o que ficou fora das análises e por quê.

## 8. Separar a base adequada para a análise

Nesta aula usaremos uma base única, mais restrita: registros válidos e com status Concluído. Essa escolha evita comparar resultados com bases diferentes sem perceber. Como exige custo e data preenchidos, deixa de fora alguns pedidos que poderiam entrar em um cálculo isolado de receita; é uma decisão didática e precisa ser documentada.

Pedidos cancelados ou devolvidos não entrarão na receita e na margem. Esta é uma simplificação da atividade, não um modelo contábil de devoluções. Os dados não contêm datas de estorno nem despesas operacionais.

In [9]:
df_analise = df_org.loc[
    df_org["registro_valido"]
    & (df_org["status_pedido"] == "Concluído")
].copy()

print("Pedidos distintos registrados:", len(df_org))
print("Pedidos com algum problema:", (~df_org["registro_valido"]).sum())
print("Pedidos concluídos válidos para análise:", len(df_analise))
print("Status registrados (inclui pedidos com problemas):")
print(df_org["status_pedido"].value_counts())

Pedidos distintos registrados: 1500
Pedidos com algum problema: 30
Pedidos concluídos válidos para análise: 1368
Status registrados (inclui pedidos com problemas):
status_pedido
Concluído    1394
Cancelado      74
Devolvido      32
Name: count, dtype: int64


## 9. Calcular receita e margem bruta simulada

Para cada pedido:

- Valor antes do desconto = quantidade × preço unitário.
- Valor do desconto = valor antes do desconto × percentual / 100.
- Receita após desconto = valor antes do desconto − valor do desconto.
- Custo dos produtos = quantidade × custo unitário.
- Margem bruta em reais = receita após desconto − custo dos produtos.

A margem calculada **não é lucro líquido**: não desconta impostos, frete, aluguel, salários nem taxas. Aqui também não representa uma demonstração contábil completa. Um item que gera mais receita pode gerar menos margem.

O Pandas realiza essas operações em cada linha sem escrevermos um `for`. As colunas novas são atribuídas usando colchetes e `=`.

In [10]:
df_analise["valor_bruto"] = df_analise["quantidade"] * df_analise["preco_unitario"]
df_analise["valor_desconto"] = df_analise["valor_bruto"] * df_analise["desconto_pct"] / 100
df_analise["receita"] = df_analise["valor_bruto"] - df_analise["valor_desconto"]
df_analise["custo_produtos"] = df_analise["quantidade"] * df_analise["custo_unitario"]
df_analise["margem_bruta"] = df_analise["receita"] - df_analise["custo_produtos"]

print(df_analise[["pedido_id", "produto", "quantidade", "receita", "margem_bruta"]].head())
print(df_analise["receita"].describe())

   pedido_id           produto  quantidade  receita  margem_bruta
0       1359      Barra amarga           2   19.822         3.502
1        550    Barra ao leite           3   24.000         6.600
2        654  Caixa de bombons           1   24.000         8.500
3       1123      Barra amarga           4   41.976         9.336
4        674  Caixa de bombons           6  108.000        15.000
count    1368.000000
mean       92.257086
std        94.417636
min         6.400000
25%        25.440000
50%        58.300000
75%       124.020000
max       704.900000
Name: receita, dtype: float64


`describe()` resume valores numéricos. `count` é a quantidade de valores preenchidos; `mean` é a média; `min` e `max` são os extremos; `50%` é a mediana. A mediana é o valor central depois de ordenar. Não é necessário dominar quartis e desvio padrão para começar esta aula.

**Valor médio por pedido**, ou ticket médio nesta atividade, é receita total dividida pela quantidade de pedidos concluídos válidos. É diferente de preço médio por produto ou de quantidade média de unidades.

In [11]:
print("Receita após desconto: R$", round(df_analise["receita"].sum(), 2))
print("Margem bruta simulada: R$", round(df_analise["margem_bruta"].sum(), 2))
print("Unidades:", df_analise["quantidade"].sum())
print("Valor médio por pedido: R$", round(df_analise["receita"].mean(), 2))
print("Mediana do valor por pedido: R$", round(df_analise["receita"].median(), 2))

Receita após desconto: R$ 126207.69
Margem bruta simulada: R$ 42481.25
Unidades: 4266
Valor médio por pedido: R$ 92.26
Mediana do valor por pedido: R$ 58.3


## 10. Comparar categorias e canais

`groupby()` separa linhas em grupos. `agg()` calcula medidas em cada grupo. Na sintaxe `receita=("receita", "sum")`, o nome à esquerda será a coluna da saída; os dois itens entre parênteses indicam coluna de origem e cálculo.

Vamos contar pedidos com `size`, somar unidades, receita e margem, e calcular média da receita por pedido. Uma contagem de pedidos não substitui uma soma de unidades.

In [12]:
resumo_categoria = df_analise.groupby("categoria").agg(
    pedidos=("pedido_id", "size"),
    unidades=("quantidade", "sum"),
    receita=("receita", "sum"),
    margem_bruta=("margem_bruta", "sum"),
)
print(resumo_categoria.round(2))

resumo_canal = df_analise.groupby("canal_venda").agg(
    pedidos=("pedido_id", "size"),
    unidades=("quantidade", "sum"),
    receita=("receita", "sum"),
    margem_bruta=("margem_bruta", "sum"),
    valor_medio_pedido=("receita", "mean"),
)
resumo_canal["margem_pct"] = 100 * resumo_canal["margem_bruta"] / resumo_canal["receita"]
print(resumo_canal.round(2))

           pedidos  unidades   receita  margem_bruta
categoria                                           
Barra          571      1934  17025.93       4022.15
Caixa          316      1093  25409.95       7929.05
Ovo            481      1239  83771.82      30530.06
             pedidos  unidades  ...  valor_medio_pedido  margem_pct
canal_venda                     ...                                
Loja física      382       751  ...               78.92       34.35
Mercado          594      2733  ...              106.17       30.61
Web              392       782  ...               84.18       38.87

[3 rows x 6 columns]


`margem_pct` é calculada usando **margem total / receita total** do grupo. Não é a média simples dos percentuais de pedidos individuais.

**Converse com a turma:** o canal que lidera em unidades também lidera em receita? A diferença pode estar relacionada ao mix de produtos, ao preço, ao desconto e à quantidade. Não conclua que o canal sozinho causou um resultado.

### Cruzar produto e canal

Precisamos olhar a combinação, não apenas totais separados. `groupby(["canal_venda", "categoria"])` agrupa por duas colunas. `unstack(fill_value=0)` organiza uma delas em colunas para facilitar a comparação. Zero aqui significa que não houve unidades observadas naquela combinação na base analisada.

In [13]:
unidades_canal_categoria = (
    df_analise.groupby(["canal_venda", "categoria"])["quantidade"]
    .sum().unstack(fill_value=0)
)
print(unidades_canal_categoria)

participacao = unidades_canal_categoria.div(
    unidades_canal_categoria.sum(axis=1), axis=0
) * 100
print("Participação de cada categoria nas unidades do próprio canal (%):")
print(participacao.round(1))

categoria    Barra  Caixa  Ovo
canal_venda                   
Loja física    261    118  372
Mercado       1405    830  498
Web            268    145  369
Participação de cada categoria nas unidades do próprio canal (%):
categoria    Barra  Caixa   Ovo
canal_venda                    
Loja física   34.8   15.7  49.5
Mercado       51.4   30.4  18.2
Web           34.3   18.5  47.2


A participação usa como denominador as unidades de **cada canal**. Comparar apenas quantidades absolutas pode refletir o tamanho do canal. Os percentuais ajudam a observar seu mix, sem provar preferência de consumidores reais.

## 11. Explorar o calendário e a sazonalidade

Sazonalidade é um padrão associado a períodos recorrentes, como uma data comemorativa. Vamos comparar receita por ano-mês, preservando a ordem temporal. Não temos julho a dezembro: não preencha esses meses como vendas zero, porque eles não fazem parte da cobertura.

Não confunda receita maior com mais unidades: preços, mix e descontos também alteram a receita.

In [14]:
resumo_mensal = df_analise.groupby("ano_mes").agg(
    pedidos=("pedido_id", "size"),
    unidades=("quantidade", "sum"),
    receita=("receita", "sum"),
).sort_index()
print(resumo_mensal.round(2))

resumo_ano = df_analise.groupby("ano").agg(
    pedidos=("pedido_id", "size"),
    unidades=("quantidade", "sum"),
    receita=("receita", "sum"),
)
print("Comparação janeiro–junho, não ano completo:")
print(resumo_ano.round(2))

         pedidos  unidades   receita
ano_mes                             
2024-01       43       124   2209.20
2024-02       53       140   2743.70
2024-03      332      1016  35698.80
2024-04      150       476  12308.75
2024-05       46       141   1867.35
2024-06       51       147   2949.05
2025-01       39       129   2252.82
2025-02       39       121   1796.86
2025-03      147       452  13389.71
2025-04      378      1259  42812.76
2025-05       59       164   5946.34
2025-06       31        97   2232.36
Comparação janeiro–junho, não ano completo:
      pedidos  unidades   receita
ano                              
2024      675      2044  57776.85
2025      693      2222  68430.84


### Comparar a mesma distância da Páscoa

Como a data muda, vamos calcular quantos dias o pedido está distante da Páscoa daquele ano. Negativo significa antes; zero significa no dia; positivo significa depois.

`map()` relaciona o ano com a data da Páscoa. `dt.normalize()` remove a hora para comparar dias de calendário; sem isso, uma venda no dia anterior poderia ser classificada incorretamente por uma diferença de horas. A subtração gera uma duração; `.dt.days` extrai dias.

Separaremos três janelas de mesmo tamanho: 30 dias antes, 30 dias depois (incluindo o dia da Páscoa) e demais dias. As janelas comparadas existem integralmente nos dois anos.

In [15]:
datas_pascoa = {
    2024: pd.Timestamp("2024-03-31"),
    2025: pd.Timestamp("2025-04-20"),
}
df_analise["data_pascoa"] = pd.to_datetime(df_analise["ano"].map(datas_pascoa))
df_analise["dias_da_pascoa"] = (
    df_analise["data_venda"].dt.normalize() - df_analise["data_pascoa"]
).dt.days

df_analise["janela_pascoa"] = "Demais dias"
df_analise.loc[df_analise["dias_da_pascoa"].between(-30, -1), "janela_pascoa"] = "30 dias antes"
df_analise.loc[df_analise["dias_da_pascoa"].between(0, 29), "janela_pascoa"] = "30 dias depois"

janelas = df_analise.groupby(["ano", "janela_pascoa"]).agg(
    pedidos=("pedido_id", "size"),
    unidades=("quantidade", "sum"),
    receita=("receita", "sum"),
)
print(janelas.round(2))

                     pedidos  unidades   receita
ano  janela_pascoa                              
2024 30 dias antes       328      1002  35319.80
     30 dias depois      154       490  12687.75
     Demais dias         193       552   9769.30
2025 30 dias antes       360      1180  41100.55
     30 dias depois      154       481  16757.12
     Demais dias         179       561  10573.18


**Interprete com cuidado:** a categoria “Demais dias” tem mais dias de cobertura que cada janela de 30 dias. Não compare esses totais como se os períodos tivessem a mesma duração. Compare as duas janelas de 30 dias entre si e a mesma janela entre anos.

Os padrões são intencionais nesta simulação. Isso ensina a investigar sazonalidade, mas não constitui evidência sobre o comércio real.

## 12. Investigar status e horário — extensão opcional

Para taxa de cancelamento, o denominador deve incluir pedidos cancelados. Usar somente `df_analise`, que contém concluídos, produziria uma taxa artificial de zero.

Usaremos a base organizada, sem duplicatas, com canais conhecidos e status reconhecidos. Aqui não exigimos preço ou custo preenchido, pois a pergunta é sobre a situação do pedido. Sempre informe a base de cada medida.

In [16]:
base_status = df_org.loc[
    df_org["canal_venda"].notna()
    & df_org["status_pedido"].isin(status_validos)
].copy()
base_status["cancelado"] = base_status["status_pedido"] == "Cancelado"
base_status["devolvido"] = base_status["status_pedido"] == "Devolvido"
resumo_status = base_status.groupby("canal_venda").agg(
    pedidos_registrados=("pedido_id", "size"),
    cancelados=("cancelado", "sum"),
    devolvidos=("devolvido", "sum"),
)
resumo_status["taxa_cancelamento_pct"] = (
    100 * resumo_status["cancelados"] / resumo_status["pedidos_registrados"]
)
print(resumo_status.round(2))

print("Pedidos concluídos válidos por canal e hora:")
print(df_analise.groupby(["canal_venda", "hora"]).size().unstack(fill_value=0))

             pedidos_registrados  cancelados  devolvidos  taxa_cancelamento_pct
canal_venda                                                                    
Loja física                  405          11           6                   2.72
Mercado                      641          26          12                   4.06
Web                          454          37          14                   8.15
Pedidos concluídos válidos por canal e hora:
hora         9   10  11  12  13  14  15  16  17  18  19  20  21  22
canal_venda                                                        
Loja física  36  39  24  32  31  37  46  37  34  34  32   0   0   0
Mercado      61  49  44  45  43  72  58  56  52  56  58   0   0   0
Web           0   0   0   0   0   0   0   0   0  62  89  83  82  76


Uma diferença de taxa pode motivar investigar pagamento, prazo, estoque ou experiência de compra. Esses motivos não estão no arquivo: não podemos afirmar qual deles explica o resultado.

A hora é a do registro do pedido, não a hora de entrega. Como o horário local já é igual para todos nesta atividade, podemos comparar as horas diretamente.

## 13. De resultado a insight

Um resultado descreve um número. Um insight conecta uma evidência a uma pergunta e a uma implicação, com limites explícitos.

Use esta estrutura:

1. **Pergunta:** o que queríamos descobrir?
2. **Evidência:** quais valores e qual base sustentam a resposta?
3. **Interpretação:** o que a diferença sugere nesta simulação?
4. **Próximo passo:** o que a gestão poderia investigar ou testar?
5. **Limitação:** o que os dados não permitem concluir?

Exemplo de redação, para preencher com valores reais das tabelas: “No canal ___, a categoria ___ representa ___% das unidades concluídas válidas. Isso sugere adequar o mix desse canal. Antes de alterar estoque, seria necessário verificar disponibilidade e pedidos não atendidos, que não aparecem no CSV.”

Não escreva “aumentar o estoque vai aumentar as vendas”: o dataset não demonstra esse efeito. Também não chame unidades de barra e de ovo de quantidade igual de chocolate; os pesos são diferentes.

## 14. Exercícios — tente antes de consultar o gabarito

Use a base indicada em cada pergunta. Após cada código, acrescente uma célula de texto para interpretar a resposta. É possível gerar insights sem gráficos.

### Exercício 1

Na base organizada, conte pedidos registrados por ano e mês. Explique por que não agrupar apenas por mês.

In [17]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 2

Na base de análise, qual produto lidera em unidades? E em margem bruta total? As respostas coincidem?

In [18]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 3

Selecione pedidos concluídos válidos da Web em abril de 2025. Mostre ID, data, produto e receita.

In [19]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 4

Calcule o valor médio por pedido por canal. Compare com a mediana e explique a diferença.

In [20]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 5

Compare unidades por categoria nos 30 dias antes e nos 30 dias depois da Páscoa, separando os anos.

In [21]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 6

Compare quantidade média em pedidos com e sem desconto. Explique por que isso não prova que o desconto causou mais vendas.

In [22]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 7

Calcule o peso total vendido, em quilos, por categoria. Por que unidades e peso são medidas diferentes?

In [23]:
# Escreva sua solução aqui.

**Minha interpretação:** ...

### Exercício 8 — três insights

Escolha três descobertas diferentes: uma temporal, uma sobre produto/canal e uma sobre receita/margem ou status. Escreva evidência numérica, interpretação, próximo passo e limitação. Não basta repetir “a categoria X vendeu mais”.

**Insight 1:** ...

**Insight 2:** ...

**Insight 3:** ...

## 15. Exportar, documentar e conferir

Exportaremos dois arquivos: a base organizada, incluindo flags e problemas, e a base efetivamente usada nos cálculos. A segunda permite conferir receita e margem; a primeira permite auditar as exclusões.

`mkdir()` cria a pasta. `parents=True` permite criar pastas intermediárias; `exist_ok=True` permite que a pasta já exista. `index=False` evita gravar o índice automático como uma coluna adicional. `date_format` controla a apresentação de datetime no CSV, que continuará sendo um arquivo de texto e precisará ser convertido novamente numa leitura futura.

In [24]:
Path("dados_organizados").mkdir(parents=True, exist_ok=True)
df_org.to_csv("dados_organizados/pascoa_organizada.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
df_analise.to_csv("dados_organizados/pascoa_base_analise.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")

conferencia = pd.read_csv("dados_organizados/pascoa_base_analise.csv")
print("Linhas exportadas:", len(conferencia))
print("Receita conferida:", round(conferencia["receita"].sum(), 2))

Linhas exportadas: 1368
Receita conferida: 126207.69


Reinicie o kernel e execute tudo em ordem. O projeto deve funcionar sem depender de uma célula antiga ou de uma variável criada fora do roteiro. Registre fonte, decisões, exclusões, fórmulas e limites no README e nas células de texto. Use commits reais para registrar etapas, como leitura, diagnóstico e análises.

### Dificuldades frequentes

| Mensagem ou situação | O que conferir |
|---|---|
| FileNotFoundError | CSV ao lado do notebook e pasta de execução |
| NameError | Células anteriores executadas no mesmo kernel |
| KeyError | Nome exato da coluna em `df.columns` |
| Erro ao usar `.dt` | Conversão para datetime foi executada? |
| Tudo em uma coluna | Separador do CSV; neste arquivo, vírgula |
| Resultado não muda | Reexecutar célula editada e dependentes |
| Ausências após conversão | Mostrar os valores originais e investigar |

**Fechamento:** qual foi a pergunta? Que base usamos? Que evidência encontramos? O que a gestão poderia investigar? O que permanece desconhecido?

## 16. Gabarito executável — consultar depois de tentar

A professora pode usar estas soluções para conferência. Outras abordagens corretas são possíveis.

### Solução 1

Na base organizada, conte pedidos registrados por ano e mês. Explique por que não agrupar apenas por mês.

In [25]:
print(df_org.dropna(subset=["data_venda"]).groupby(["ano", "mes"]).size())

ano   mes
2024  1       45
      2       57
      3      371
      4      170
      5       50
      6       55
2025  1       42
      2       43
      3      157
      4      402
      5       70
      6       34
dtype: int64


### Solução 2

Na base de análise, qual produto lidera em unidades? E em margem bruta total? As respostas coincidem?

In [26]:
r = df_analise.groupby("produto").agg(unidades=("quantidade", "sum"), margem=("margem_bruta", "sum"))
print(r.sort_values("unidades", ascending=False).round(2))
print(r.sort_values("margem", ascending=False).round(2))

                  unidades    margem
produto                             
Barra ao leite        1197   2088.04
Caixa de bombons      1093   7929.05
Barra amarga           737   1934.11
Ovo tradicional        469   6940.82
Ovo infantil           396   6636.07
Ovo premium            374  16953.16
                  unidades    margem
produto                             
Ovo premium            374  16953.16
Caixa de bombons      1093   7929.05
Ovo tradicional        469   6940.82
Ovo infantil           396   6636.07
Barra ao leite        1197   2088.04
Barra amarga           737   1934.11


### Solução 3

Selecione pedidos concluídos válidos da Web em abril de 2025. Mostre ID, data, produto e receita.

In [27]:
selecao = (df_analise["canal_venda"] == "Web") & (df_analise["ano"] == 2025) & (df_analise["mes"] == 4)
print(df_analise.loc[selecao, ["pedido_id", "data_venda", "produto", "receita"]].head(10))
print("Quantidade de pedidos:", selecao.sum())

     pedido_id          data_venda           produto  receita
9          915 2025-04-04 20:19:56      Ovo infantil  137.800
14         940 2025-04-20 20:44:38    Barra ao leite    8.056
25        1463 2025-04-13 22:00:25  Caixa de bombons   76.320
37        1293 2025-04-25 18:58:09       Ovo premium  226.575
38        1291 2025-04-14 18:40:44       Ovo premium  302.100
51        1068 2025-04-28 22:50:40   Ovo tradicional   87.450
98        1484 2025-04-16 20:40:56    Barra ao leite   24.168
100        791 2025-04-16 22:09:08      Ovo infantil   68.900
160       1372 2025-04-12 19:35:00  Caixa de bombons   25.440
184        993 2025-04-11 19:45:21    Barra ao leite   25.440
Quantidade de pedidos: 91


### Solução 4

Calcule o valor médio por pedido por canal. Compare com a mediana e explique a diferença.

In [28]:
print(df_analise.groupby("canal_venda")["receita"].agg(["mean", "median"]).round(2))

               mean  median
canal_venda                
Loja física   78.92   57.42
Mercado      106.17   58.83
Web           84.18   55.00


### Solução 5

Compare unidades por categoria nos 30 dias antes e nos 30 dias depois da Páscoa, separando os anos.

In [29]:
b = df_analise.loc[df_analise["janela_pascoa"] != "Demais dias"]
print(b.groupby(["ano", "janela_pascoa", "categoria"])["quantidade"].sum())

ano   janela_pascoa   categoria
2024  30 dias antes   Barra        386
                      Caixa        219
                      Ovo          397
      30 dias depois  Barra        252
                      Caixa         95
                      Ovo          143
2025  30 dias antes   Barra        489
                      Caixa        283
                      Ovo          408
      30 dias depois  Barra        148
                      Caixa        145
                      Ovo          188
Name: quantidade, dtype: int64


### Solução 6

Compare quantidade média em pedidos com e sem desconto. Explique por que isso não prova que o desconto causou mais vendas.

In [30]:
b = df_analise.copy()
b["com_desconto"] = b["desconto_pct"] > 0
print(b.groupby("com_desconto").agg(pedidos=("pedido_id", "size"), quantidade_media=("quantidade", "mean")).round(2))

              pedidos  quantidade_media
com_desconto                           
False             655              3.09
True              713              3.14


### Solução 7

Calcule o peso total vendido, em quilos, por categoria. Por que unidades e peso são medidas diferentes?

In [31]:
b = df_analise.copy()
b["peso_total_kg"] = b["quantidade"] * b["peso_g"] / 1000
print(b.groupby("categoria")["peso_total_kg"].sum().round(2))

categoria
Barra    174.06
Caixa    273.25
Ovo      307.55
Name: peso_total_kg, dtype: float64


### Conferência geral e exemplos de conclusões para a professora

- Original: 1.515 linhas; após duplicatas: 1.500 pedidos distintos.
- Pedidos com algum problema nas regras: 30.
- Base de análise: 1368 pedidos concluídos válidos, 4266 unidades, R$ 126.207,69 de receita após desconto e R$ 42.481,25 de margem bruta simulada.
- Produto líder em unidades: Barra ao leite, com 1197 unidades. Produto líder em margem: Ovo premium, com R$ 16.953,17. Isso mostra que volume e contribuição em reais podem ter líderes diferentes; priorização exige também estoque e demanda não atendida, ausentes no arquivo.
- Valor médio por pedido: Web R$ 84,18; Mercado R$ 106,17. A diferença sugere investigar o mix e a quantidade por pedido; não prova um efeito causal do canal.
- Cancelamento: Web 8,15%; Mercado 4,06%. A base desta taxa inclui todos os status reconhecidos. O próximo passo seria investigar motivos; o arquivo não informa esses motivos.

Os resultados são da simulação, não recomendações comprovadas para empresas reais. A seção de exercícios sobre tempo deve considerar o calendário da Páscoa e as janelas iguais, além dos totais mensais.